# Pipeline de Entrenamiento

_Cerrar el ciclo: del feature store al modelo de regresion entrenado y evaluado, con consistencia entrenamiento-serving._

**Modulo 1 — Feature Engineering & Feature Stores** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![Pipeline de Entrenamiento](assets/header.png)


## Introduccion

En el Notebook 3 definimos features en **Feast** y aprendimos a materializarlas y
servirlas. Pero un feature store no es un fin en si mismo: existe para
**alimentar modelos**. En este notebook **cerramos el ciclo**:

```
   feature store  ->  set de entrenamiento  ->  modelo de regresion entrenado y evaluado
```

Es decir:

1. Construimos el **set de entrenamiento** leyendo del **offline store** de Feast
   con `get_historical_features` (con un fallback a parquet si Feast no esta
   disponible).
2. Entrenamos un **regresor** que predice `SalePrice` (entrenado sobre
   `log1p(SalePrice)`, como en la competencia de Kaggle).
3. Lo **evaluamos** con **RMSE, MAE y R²**, ademas del **RMSE sobre `log(SalePrice)`**
   (la metrica oficial de Kaggle), y graficamos **predicho vs real**.

> **La idea grande: consistencia entrenamiento-serving.** El set de entrenamiento
> sale de las *mismas* definiciones de features que luego serviran online. Por eso
> el modelo ve en produccion exactamente la misma representacion con la que
> aprendio: no hay sorpresas de *skew*. El feature store es el contrato que une
> ambos mundos.


## Setup

Definimos las rutas y un helper para localizar el repo de Feast y el parquet del
offline store. No se necesita ningun servicio levantado para la ruta de parquet;
para usar Feast directamente, primero corre `feast apply` (Notebook 3).

In [ ]:
import os
from datetime import datetime
import numpy as np
import pandas as pd

REPO = os.path.abspath(os.path.join("..", "platform", "feature_repo"))
PARQUET_PATH = os.path.join(REPO, "data", "housing_features.parquet")
print("repo de features:", REPO)
print("parquet offline :", PARQUET_PATH)
print("existe parquet  :", os.path.exists(PARQUET_PATH))

## Paso 1 — construir el set de entrenamiento desde el feature store

La forma *correcta* de armar un set de entrenamiento es con
`get_historical_features`: le pasas un **dataframe de entidades** (que casas y
en que `event_timestamp`) y te devuelve las features validas en ese momento
(point-in-time). Aqui usamos como timestamp "ahora", asi que traemos el ultimo
valor de cada casa.

Si Feast no esta instalado o el registry no esta aplicado, caemos a **leer el
parquet del offline store directamente** (la misma fuente que Feast usa por
debajo). En ambos casos obtenemos las mismas features.

In [ ]:
def load_offline_parquet():
    """Fallback: lee directamente el parquet del offline store de Feast."""
    df = pd.read_parquet(PARQUET_PATH)
    print(f"[parquet] {len(df)} filas leidas de {PARQUET_PATH}")
    return df

FEATURES = [
    "house_features:overall_qual",
    "house_features:gr_liv_area",
    "house_features:total_bsmt_sf",
    "house_features:first_flr_sf",
    "house_features:garage_cars",
    "house_features:garage_area",
    "house_features:year_built",
    "house_features:lot_area",
    "house_features:full_bath",
    "house_features:neighborhood",
    "house_features:exter_qual_ord",
]

def build_training_set():
    """Intenta Feast (get_historical_features); si falla, usa el parquet."""
    try:
        from feast import FeatureStore
        store = FeatureStore(repo_path=REPO)

        base = pd.read_parquet(PARQUET_PATH)  # de aqui sacamos entidades + objetivo
        entity_df = pd.DataFrame({
            "house_id": base["house_id"].values,
            "event_timestamp": [datetime.utcnow()] * len(base),
        })
        feats = store.get_historical_features(
            entity_df=entity_df, features=FEATURES,
        ).to_df()
        # Adjuntamos el objetivo `sale_price` desde el parquet (no es una feature servida).
        feats = feats.merge(base[["house_id", "sale_price"]], on="house_id")
        print(f"[feast] set de entrenamiento point-in-time: {feats.shape}")
        return feats
    except Exception as e:
        print(f"[feast] no disponible ({type(e).__name__}: {e}); uso el parquet.")
        return load_offline_parquet()

train_df = build_training_set()
train_df.head()

## Paso 2 — entrenar un regresor

Predecimos `SalePrice` con un **`RandomForestRegressor`** dentro de un `Pipeline`
de sklearn. Entrenamos sobre **`log1p(SalePrice)`** (asi el modelo optimiza el error
en escala logaritmica, que es la metrica de Kaggle y reduce el peso de las casas
caras). El pipeline encapsula el preprocesamiento (one-hot del barrio) junto con el
modelo, de modo que *exactamente los mismos pasos* corren en entrenamiento y en
serving. Eso es, de nuevo, consistencia entrenamiento-serving, ahora a nivel de
codigo del modelo.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric = ["overall_qual", "gr_liv_area", "total_bsmt_sf", "first_flr_sf",
           "garage_cars", "garage_area", "year_built", "lot_area",
           "full_bath", "exter_qual_ord"]
categorical = ["neighborhood"]

# Solo conservamos columnas que existan (robusto al fallback de parquet).
numeric = [c for c in numeric if c in train_df.columns]
categorical = [c for c in categorical if c in train_df.columns]

X = train_df[numeric + categorical].copy()
y = train_df["sale_price"].astype(float)
y_log = np.log1p(y)                       # entrenamos en escala logaritmica

X_train, X_test, y_train, y_test, ylog_train, ylog_test = train_test_split(
    X, y, y_log, test_size=0.25, random_state=42,
)

pre = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), categorical)],
    remainder="passthrough",
)
model = Pipeline(steps=[
    ("pre", pre),
    ("reg", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
])

model.fit(X_train, ylog_train)            # objetivo = log1p(SalePrice)
print("Modelo de regresion entrenado. Features:", numeric + categorical)

## Paso 3 — evaluar

Medimos las metricas tipicas de regresion sobre el precio en su escala original
(en dolares): **RMSE** (raiz del error cuadratico medio), **MAE** (error absoluto
medio) y **R²** (fraccion de varianza explicada). Ademas reportamos el **RMSE sobre
`log(SalePrice)`**, que es la **metrica oficial de la competencia de Kaggle** (y la
que el modelo optimiza directamente). Cerramos con un grafico de **predicho vs
real**.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Predecimos en escala log y volvemos a dolares con expm1.
pred_log = model.predict(X_test)
pred = np.expm1(pred_log)

rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
rmse_log = np.sqrt(mean_squared_error(ylog_test, pred_log))   # metrica de Kaggle

print(f"RMSE  (USD)          : {rmse:,.0f}")
print(f"MAE   (USD)          : {mae:,.0f}")
print(f"R2                   : {r2:.4f}")
print(f"RMSE sobre log(price): {rmse_log:.4f}   <- metrica oficial de Kaggle")

In [ ]:
import matplotlib.pyplot as plt

# Predicho vs real: cuanto mas pegados a la diagonal, mejor.
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_test, pred, s=14, alpha=0.5, c="#4c78a8")
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
ax.plot(lims, lims, "--", c="#e45756", lw=1.5, label="prediccion perfecta")
ax.set_xlabel("SalePrice real"); ax.set_ylabel("SalePrice predicho")
ax.set_title(f"Predicho vs real (R²={r2:.3f})")
ax.legend(); plt.tight_layout(); plt.show()

## Paso 4 — persistir el modelo

Antes de servir predicciones, **persistimos el modelo entrenado** en disco para
reutilizarlo (lo cargaremos tal cual en el paso de serving, como haria un servicio
de inferencia):

In [ ]:
import joblib

MODEL_PATH = os.path.join(REPO, "data", "housing_model.joblib")
joblib.dump(model, MODEL_PATH)
print(f"Modelo guardado en: {MODEL_PATH}")
print(f"Metricas -> RMSE={rmse:,.0f}  MAE={mae:,.0f}  R2={r2:.4f}  RMSE_log={rmse_log:.4f}")

## Paso 5 — serving online: inferencia con features de Feast

Entrenamos con features **historicas** (`get_historical_features`). En **produccion**,
cuando llega una peticion —"¿cuanto vale *esta* casa?"— no recalculamos todo el
historico: le pedimos al **online store** (Redis) el **ultimo** valor de cada feature
para esa entidad con **`get_online_features`** y se lo pasamos al modelo.

Este es el cierre real del ciclo: **las mismas definiciones de features** que
armaron el set de entrenamiento ahora **sirven la inferencia**, con latencia de
milisegundos y sin sesgo entrenamiento-serving.

```
   peticion (house_id) -> get_online_features (Redis) -> modelo.predict -> SalePrice
```

> Requiere haber corrido `feast apply` + `feast materialize` (Notebook 3) y tener
> Redis arriba. Si el online store no esta disponible, caemos a tomar la fila local
> para que el ejemplo siga siendo ejecutable.

In [ ]:
# Cargamos el modelo persistido, como lo haria un servicio de inferencia.
served_model = joblib.load(MODEL_PATH)

# Elegimos una casa cualquiera para simular una peticion de serving.
example_id = int(train_df["house_id"].iloc[0])

def features_from_online_store(house_id):
    """Lee las ULTIMAS features de la casa desde el online store de Feast (Redis)."""
    from feast import FeatureStore
    store = FeatureStore(repo_path=REPO)
    refs = [f"house_features:{c}" for c in (numeric + categorical)]
    online = store.get_online_features(
        features=refs, entity_rows=[{"house_id": house_id}],
    ).to_dict()
    # to_dict() devuelve las columnas por su nombre (sin el prefijo "house_features:").
    return pd.DataFrame({c: online[c] for c in (numeric + categorical)})

try:
    X_one = features_from_online_store(example_id)
    fuente = "online store de Feast (Redis)"
except Exception as e:
    # Fallback ejecutable si Feast/Redis no estan arriba.
    print(f"[serving] online store no disponible ({type(e).__name__}); uso una fila local.")
    X_one = train_df.loc[train_df["house_id"] == example_id, numeric + categorical]
    fuente = "fila local (fallback)"

# El modelo es un Pipeline: aplica el MISMO preprocesamiento del entrenamiento.
pred_price = float(np.expm1(served_model.predict(X_one)[0]))   # de log a dolares
print(f"Casa house_id={example_id} | features desde: {fuente}")
print(f"Prediccion de SalePrice: ${pred_price:,.0f}")

## Repaso — el ciclo cerrado

Acabamos de recorrer el ciclo completo de un sistema de ML del mundo real:

1. **Feature store (Feast)** define y sirve features de forma consistente.
2. `get_historical_features` arma un **set de entrenamiento** point-in-time
   correcto (sin leakage).
3. Un **modelo de regresion** se entrena con ese set, dentro de un pipeline que
   garantiza que el preprocesamiento sera identico en serving.
4. El modelo se **evalua** (RMSE, MAE, R² y el RMSE sobre log(price) de Kaggle) y
   se **persiste** en disco.
5. En **serving online**, `get_online_features` lee las ultimas features desde
   Redis y el modelo predice en tiempo real — cerrando el ciclo train → serve.

**Por que importa la consistencia entrenamiento-serving:** el modelo aprende sobre
una representacion concreta de los datos. Si en produccion las features se
calculan de otra forma (otra mediana de imputacion, otro escalado, otra version de
la transformacion), el modelo recibe entradas "fuera de distribucion" y se degrada
en silencio. El feature store + el pipeline de sklearn son las dos piezas que
garantizan que *lo que se entrena es lo que se sirve*.

**Orquestacion.** Este mismo ciclo lo automatiza el DAG de **Apache Airflow** de la
plataforma compartida (`platform/dags/feature_engineering_dag.py`):
`extract -> prepare -> transform -> validate -> feast_apply -> feast_materialize ->
train_model`. Airflow es el **orquestador compartido del curso**, introducido en
este modulo.